# This notebook runs the quantitative analysis
Please provide the right configuration and run the notebook through to the end

In [ ]:
import logging
import os
import sys
import torch
import yaml

from functools import partial
from logging.handlers import RotatingFileHandler
from jsonschema import validate, ValidationError

from augment import SpecAugment, AudioAugment
from esc_audio_dataset import ESCAudioDataset
from bcos.modules.bcosconv2d import BcosConv2d
from bcos_explain_shell import BcosExplainShell
from config.config_validation_template import CONFIG_TEMPLATE
from custom_logger_formatter import CustomLoggerFormatter
from gradcam import GradCAM, _get_gradcam_target_layer
from main import DATASET_MAPPING
from lime_explainer import LimeExplainer
from resnet_bcos import make_resnet18, make_resnet34, make_resnet50
from resnet_18_baseline import BaselineResNet
from tune import TUNABLE_PARAMS

import qual_analysis_utils as q_utils

### 0. Config & setup

In [ ]:
EXPERIMENT_DIR = os.path.join("output", "hypertune_baseline") # change directory to the specific experiment

BEST_MODEL_DIR = os.path.join("job_0", "best_model") # folder containing .pt model, relative from `EXPERIMENT_DIR`

DEVICE = "cuda" # use if CUDA or ROCm

EXPL_MODEL_FOR_BASELINE = "lime" # Can be `lime` or `gradcam`

SMOOTH_BCOS: bool = False # Use the smoothed explanation variant of bcos instead of the raw contribution map

In [ ]:
# Initialise Logger.
def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("qual logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
f_ch = RotatingFileHandler(f"{EXPERIMENT_DIR}/qualitative_analysis.log")
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the qualitative analysis process.")

In [ ]:
# validate the provided config file.
# with open(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/run_config.yml", 'r') as stream:
with open(os.path.join(EXPERIMENT_DIR, BEST_MODEL_DIR, "run_config.yml"), 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(os.path.join(EXPERIMENT_DIR, "config.yml"), 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

if "jobs" in CONFIG.keys():
    CONFIG = CONFIG["jobs"]["job0"]
for tunable_param in TUNABLE_PARAMS.keys():
    if tunable_param == "small_inputs":
        CONFIG["model_params"][tunable_param] = CONFIG["model_params"][tunable_param][0]
    elif type(CONFIG[tunable_param]) == list:
        CONFIG[tunable_param] = CONFIG[tunable_param][0]

logger.info("config: %s", CONFIG)

### 1. Load data


In [ ]:
logger.info("First making dataset with augmentations to get the right normalisation stats.")
dataset = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["train_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
    pre_transform=AudioAugment(CONFIG["sample_rate"]),
    post_transform=SpecAugment()
)
# Normalise
logger.debug("Fitting normalisation.")
dataset.fit_normalisation(list(range(len(dataset))))
mean = dataset.mean
std = dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean={dataset.mean}, std={dataset.std}"
)

logger.info("Re-constructiong train data without augmentations")
dataset = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["train_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
dataset.mean = mean
dataset.std = std

logger.debug(f"Dataset size: {len(dataset)}")
logger.debug(f"Shape of first x element: {dataset[0][0].shape}")
logger.debug(f"First y element: {dataset[0][1]}")
test_data = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["test_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
test_data.mean = mean
test_data.std = std

logger.debug(f"Test dataset size: {len(test_data)}")

### 2. Load model

In [ ]:
models = {
    "resnet18_bcos": (
        make_resnet18, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet34_bcos": (
        make_resnet34, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet50_bcos": (
        make_resnet50, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet18_baseline": (
        lambda **kwargs: BaselineResNet(kwargs["num_classes"], kwargs["logger"]),
        {"logger": logger, "num_classes": dataset.get_n_classes()}
    )   
}
model = None
for name, (cls, kwargs) in models.items():
    if CONFIG['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)


In [ ]:
model_file_path = [entry for entry in os.listdir(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}") if entry.endswith(".pth")][0]
logger.info(f"Loading model from {model_file_path}")

state_dict = torch.load(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/{model_file_path}", weights_only=True)
model.load_state_dict(state_dict)

#### 2.1. Wrapping 
This cell wraps any non b-cos model with gradcam or lime. It also can wrap a b-cos model with the smoothing shell

In [ ]:
# explains final convolutional layer
if "baseline" in CONFIG['model'].lower(): 
    if EXPL_MODEL_FOR_BASELINE.lower() == "gradcam":
        model = GradCAM(model, target_layer=_get_gradcam_target_layer(model))
        logger.info("Wrapped model in GradCam shell.")
    elif EXPL_MODEL_FOR_BASELINE.lower() == "lime":
        model = LimeExplainer(model, num_samples=2000, num_segments=64, batch_size=128)
        logger.info("Wrapped model in LIME shell.")
    else:
        logger.error(f"Unable to wrap the model, the `EXPL_MODEL_FOR_BASELINE` provided is not one of: ['gradcam', 'lime']: ({EXPL_MODEL_FOR_BASELINE})")
        raise ValueError
else:
    if SMOOTH_BCOS == True:
        model = BcosExplainShell(model)
        logger.info("Wrapped model in smoothing explanation shell.")


### 3. Convert explanations back to audio

In [ ]:
# train 
"""NOT RECOMMENDED, AS IT MAKES TOO MANY FILES!"""

# train_expl_wav_output_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/explanation_audio_train/"
# os.makedirs(train_expl_wav_output_dir, exist_ok=True)
# q_utils.back_convert_entire_dataset(
#     dataset=dataset, 
#     model=model, 
#     top_quantile=0.9, 
#     DEVICE=DEVICE, 
#     CONFIG=CONFIG, 
#     logger=logger, 
#     output_folder=train_expl_wav_output_dir
# )

# test
test_expl_wav_output_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/explanation_audio_test/"
os.makedirs(test_expl_wav_output_dir, exist_ok=True)
q_utils.back_convert_entire_dataset(
    dataset=test_data, 
    model=model, 
    top_quantile=0.9, 
    DEVICE=DEVICE, 
    CONFIG=CONFIG, 
    logger=logger, 
    output_folder=test_expl_wav_output_dir
)
logger.info("Done explaining entire test set and turning it to audio.")

### 4. Convert grid pointing examples back to audio

In [ ]:
# train 
"""NOT RECOMMENDED, AS IT MAKES TOO MANY FILES!"""

# train_qual_grid_output_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/grid_qualitative_train/"
# os.makedirs(train_qual_grid_output_dir, exist_ok=True)
# q_utils.grid_pointing_game_qual(
#     dataset=dataset, 
#     model=model, 
#     DEVICE=DEVICE,
#     CONFIG=CONFIG,
#     logger=logger,
#     top_quantile=0.9,
#     base_output_dir=train_qual_grid_output_dir
# )

# test
test_qual_grid_output_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/grid_qualitative_test/"
os.makedirs(test_qual_grid_output_dir, exist_ok=True)
q_utils.grid_pointing_game_qual(
    dataset=test_data, 
    model=model, 
    DEVICE=DEVICE,
    CONFIG=CONFIG,
    logger=logger,
    top_quantile=0.9,
    base_output_dir=test_qual_grid_output_dir
)